# Statistics

This notebook fetches and displays tabular basketball player statistics from the Basketball Stats Vlaanderen website for a specific player and season.

In [ ]:
club = "https://app.basketballstatsvlaanderen.be/clubs/BVBL1037/BVBL1037J16%20%201#stats"

In [ ]:
clubmatchesurl = "https://vblcb.wisseq.eu/VBLCB_WebService/data/OrgMatchesByGuid?issguid=BVBL1037"

In [150]:
def fetch_player_ALL(url):
    import requests
    import re
    from bs4 import BeautifulSoup
    import json
    import pandas as pd

    response = requests.get(url)
    html_doc = response.text

    soup = BeautifulSoup(html_doc, 'html.parser')
    games_data = []

    for script in soup.find_all('script'):
        if script.string and "var games = [" in script.string:
            match = re.search(r"var games = \[.*?\];", script.string, re.DOTALL)
            if match:
                games_str = match.group(0).replace('var games =', '').strip(' ;')
                try:
                    games_data = json.loads(games_str)
                    break
                except json.JSONDecodeError as e:
                    print(f"Error decoding JSON: {e}")

    # Extract subfields from 'GameTeam' if present
    for game in games_data:
        if 'GameTeam' in game and isinstance(game['GameTeam'], dict):
            game['GameTeam_TEAM'] = game['GameTeam'].get('Name', {})
            game['GameTeam_GameResult'] = game['GameTeam'].get('Game', {}).get('Result')
            game['GameTeam_GameGuid'] = game['GameTeam'].get('Game', {}).get('Guid')
            game['GameTeam_GameDate'] = game['GameTeam'].get('Game', {}).get('Date')
            game['GameTeam_AwayTeamName'] = game['GameTeam'].get('Game', {}).get('AwayTeam', {}).get('Name')
            game['GameTeam_HomeTeamName'] = game['GameTeam'].get('Game', {}).get('HomeTeam', {}).get('Name')

            # Add AGE column by trimming after the 2nd last space in GameTeam_TEAM
            for game in games_data:
                team_name = game.get('GameTeam_TEAM', '')
                if isinstance(team_name, str):
                    parts = team_name.split(' ')
                    if len(parts) > 2:
                        game['AGE'] = ' '.join(parts[-2:])
                    else:
                        game['AGE'] = team_name
                else:
                    game['AGE'] = None
    

    # filtered_games_data = [
    #     {k: game[k] for k in fields_to_keep if k in game}
    #     for game in games_data
    # ]

    df_games = pd.DataFrame(games_data)
    
    df_games = df_games.drop(columns=['id', 'GameTeamId', 'Stints', 'createdAt', 'updatedAt', 'GameTeam'])
    return df_games
fetch_player_ALL(url).iloc[0]

Guid                                                  BVBL744354
Name                                                 Vic Huysman
Number                                                         8
FunctionLetter                                                 S
Starter                                                     True
TotalMinutes                                                  33
NormalizedMinutes                                             37
FreeThrows                                                     6
FieldGoals                                                    20
ThreePointers                                                  0
TotalScore                                                    26
PlusMinus                                                      3
Faults                                                         4
Plus                                                          61
Minus                                                        -58
Badges                   

In [191]:
def FETCH_PLAYERS_AVG(url):
    df = fetch_player_ALL(url)
    # Unnest AGE from GameTeam column if not already present
    if 'AGE' not in df.columns:
        df['AGE'] = df['GameTeam'].apply(lambda x: ' '.join(x['Name'].split(' ')[-2:]) if isinstance(x, dict) and 'Name' in x else None)

    grouped = df.groupby(['Name', 'AGE'])
    result = grouped.agg({
        'TotalMinutes': 'mean',
        'NormalizedMinutes': 'mean',
        'FreeThrows': 'mean',
        'FieldGoals': 'mean',
        'ThreePointers': 'mean',
        'TotalScore': ['min', 'mean', 'median', 'max' , 'count'],
        'PlusMinus': 'mean',
        'Faults': 'mean',
        'Plus': 'mean',
        'Minus': 'mean',
       
    })

    # Flatten MultiIndex columns
    result.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in result.columns.values]
    # Rename and rearrange columns to match the desired order and names
    result = result.rename(columns={
         'TotalScore_count': 'WEDSTRIJDEN',
        
        'TotalMinutes_mean': 'Avg TotalMinutes',
        'NormalizedMinutes_mean': 'Avg NormalizedMinutes',
        'FreeThrows_mean': 'Avg FreeThrows',
        'FieldGoals_mean': 'Avg FieldGoals',
        'ThreePointers_mean': 'Avg ThreePointers',
        'TotalScore_min': 'Min TotalScore',
        'TotalScore_mean': 'Avg TotalScore',
        'TotalScore_median': 'Median TotalScore',
        'TotalScore_max': 'Max TotalScore',
       'Faults_mean': 'Avg Faults',
        'PlusMinus_mean': 'Avg PlusMinus',
        
        'Plus_mean': 'Avg Plus',
        'Minus_mean': 'Avg Minus'
    })

    # Reorder columns to match the desired template
    desired_order = [
        'WEDSTRIJDEN',
        'Avg TotalMinutes',
        'Avg NormalizedMinutes',
        'Avg FreeThrows',
        'Avg FieldGoals',
        'Avg ThreePointers',
        'Min TotalScore',
        'Avg TotalScore',
        'Median TotalScore',
        'Max TotalScore',
        
        'Avg PlusMinus',
        'Avg Faults',
        'Avg Plus',
        'Avg Minus'
    ]
    # Keep index columns (Name, AGE) at the front
    result = result.reset_index()[['Name', 'AGE'] + desired_order]
    # Round all columns containing 'Avg' to 1 decimal place
    avg_cols = [col for col in result.columns if 'Avg' in col]
    result[avg_cols] = result[avg_cols].round(1)
    return result

averages_df = FETCH_PLAYERS_AVG(url)
averages_df

,Name,AGE,WEDSTRIJDEN,Avg TotalMinutes,Avg NormalizedMinutes,Avg FreeThrows,Avg FieldGoals,Avg ThreePointers,Min TotalScore,Avg TotalScore,Median TotalScore,Max TotalScore,Avg PlusMinus,Avg Faults,Avg Plus,Avg Minus
0,Vic Huysman,J16 A,3,29.7,32.3,5.3,18.7,1.0,24,25.0,25.0,26,-29.7,2.7,49.7,-79.3
1,Vic Huysman,J18 A,4,9.5,12.0,0.2,2.5,0.0,0,2.8,3.0,5,7.5,1.2,26.5,-19.0


In [ ]:
fet

'BBC Haantjes Certifisc Oudenaarde J18 A'

## FUNCTIONS  fetch_player_summary(url)  and fetch_player(url)


In [ ]:
def fetch_player_summary(url):
    import requests
    import re
    from bs4 import BeautifulSoup
    import json
    import pandas as pd

    response = requests.get(url)
    html_doc = response.text

    soup = BeautifulSoup(html_doc, 'html.parser')
    games_data = []

    for script in soup.find_all('script'):
        if script.string and "var games = [" in script.string:
            match = re.search(r"var games = \[.*?\];", script.string, re.DOTALL)
            if match:
                games_str = match.group(0).replace('var games =', '').strip(' ;')
                try:
                    games_data = json.loads(games_str)
                    break
                except json.JSONDecodeError as e:
                    print(f"Error decoding JSON: {e}")

    fields_to_keep = [
        'Name', 'Number', 'FunctionLetter', 'Starter',
        'TotalMinutes', 'NormalizedMinutes', 'FreeThrows', 'FieldGoals', 'ThreePointers',
        'TotalScore', 'PlusMinus', 'Faults', 'Plus', 'Minus',
        'GameTeam_GameDate', 'GameTeam_GameResult', 'GameTeam_GameGuid',
        'GameTeam_HomeTeamName', 'GameTeam_AwayTeamName'
    ]

    filtered_games_data = [
        {k: game[k] for k in fields_to_keep if k in game}
        for game in games_data
    ]

    df_games = pd.DataFrame(filtered_games_data)

    fields_to_average = [
        'TotalMinutes', 'NormalizedMinutes', 'FreeThrows', 'FieldGoals', 'ThreePointers',
        'TotalScore', 'PlusMinus', 'Faults', 'Plus', 'Minus'
    ]

    summary_data = []
    for name, group in df_games.groupby('Name'):
        entry = {
            'Name': name,
            'Number': group['Number'].iloc[0] if 'Number' in group.columns else None,
            'Starter %': group['Starter'].mean() * 100 if 'Starter' in group.columns else None
        }
        for field in fields_to_average:
            filtered_group = group[group['TotalMinutes'] > 0]
            entry[field + ' Avg'] = filtered_group[field].mean() if field in filtered_group.columns else None
        summary_data.append(entry)

    summary_df = pd.DataFrame(summary_data)
    return summary_df


,Name,Number,Starter %,TotalMinutes Avg,NormalizedMinutes Avg,FreeThrows Avg,FieldGoals Avg,ThreePointers Avg,TotalScore Avg,PlusMinus Avg,Faults Avg,Plus Avg,Minus Avg
0,Vic Huysman,7,42.857143,18.142857,20.714286,2.428571,9.428571,0.428571,12.285714,-8.428571,1.857143,36.428571,-44.857143


In [129]:
fetch_player_ALL(url)

,Name,Number,Starter %,TotalMinutes Avg,NormalizedMinutes Avg,FreeThrows Avg,FieldGoals Avg,ThreePointers Avg,TotalScore Avg,PlusMinus Avg,Faults Avg,Plus Avg,Minus Avg
0,Vic Huysman,7,42.857143,18.142857,20.714286,2.428571,9.428571,0.428571,12.285714,-8.428571,1.857143,36.428571,-44.857143


In [ ]:
def fetch_player_ALL(url):
    import requests
    import re
    from bs4 import BeautifulSoup
    import json
    import pandas as pd

    response = requests.get(url)
    html_doc = response.text

    soup = BeautifulSoup(html_doc, 'html.parser')
    games_data = []

    for script in soup.find_all('script'):
        if script.string and "var games = [" in script.string:
            match = re.search(r"var games = \[.*?\];", script.string, re.DOTALL)
            if match:
                games_str = match.group(0).replace('var games =', '').strip(' ;')
                try:
                    games_data = json.loads(games_str)
                    break
                except json.JSONDecodeError as e:
                    print(f"Error decoding JSON: {e}")

    # Extract subfields from 'GameTeam' if present
    for game in games_data:
        if 'GameTeam' in game and isinstance(game['GameTeam'], dict):
            game['GameTeam_GameResult'] = game['GameTeam'].get('Game', {}).get('Result')
            game['GameTeam_GameGuid'] = game['GameTeam'].get('Game', {}).get('Guid')
            game['GameTeam_AwayTeamName'] = game['GameTeam'].get('Game', {}).get('AwayTeam', {}).get('Name')
            game['GameTeam_HomeTeamName'] = game['GameTeam'].get('Game', {}).get('HomeTeam', {}).get('Name')

    # filtered_games_data = [
    #     {k: game[k] for k in fields_to_keep if k in game}
    #     for game in games_data
    # ]

    df_games = pd.DataFrame(games_data)
    return df_games


In [87]:
def fetch_player(url):
    import requests
    import re
    from bs4 import BeautifulSoup
    import json
    import pandas as pd

    response = requests.get(url)
    html_doc = response.text

    soup = BeautifulSoup(html_doc, 'html.parser')
    games_data = []

    for script in soup.find_all('script'):
        if script.string and "var games = [" in script.string:
            match = re.search(r"var games = \[.*?\];", script.string, re.DOTALL)
            if match:
                games_str = match.group(0).replace('var games =', '').strip(' ;')
                try:
                    games_data = json.loads(games_str)
                    break
                except json.JSONDecodeError as e:
                    print(f"Error decoding JSON: {e}")

    fields_to_keep = [
        'Name', 'Number', 'FunctionLetter', 'Starter',
        'TotalMinutes', 'NormalizedMinutes', 'FreeThrows', 'FieldGoals', 'ThreePointers',
        'TotalScore', 'PlusMinus', 'Faults', 'Plus', 'Minus',

        'GameTeam',
        'GameTeam_GameDate', 'GameTeam_GameResult', 'GameTeam_GameGuid',
        'GameTeam_HomeTeamName', 'GameTeam_AwayTeamName'
    ]

    filtered_games_data = [
        {k: game[k] for k in fields_to_keep if k in game}
        for game in games_data
    ]

    df_games = pd.DataFrame(filtered_games_data)

    
    return df_games


# execute function

In [94]:
data = fetch_player(url)
data

,Name,Number,FunctionLetter,Starter,TotalMinutes,NormalizedMinutes,FreeThrows,FieldGoals,ThreePointers,TotalScore,PlusMinus,Faults,Plus,Minus,GameTeam
0,Vic Huysman,7,S,False,10,11,0,4,0,4,27,1,35,-8,"{'id': 2329102, 'Guid': 'BVBL1037J18 1', 'Nam..."
1,Vic Huysman,8,S,True,34,38,5,20,0,25,-58,1,39,-97,"{'id': 1228435, 'Guid': 'BVBL1037J16 1', 'Nam..."
2,Vic Huysman,7,S,False,15,18,1,4,0,5,-1,2,34,-35,"{'id': 1273562, 'Guid': 'BVBL1037J18 1', 'Nam..."
3,Vic Huysman,8,S,True,33,37,6,20,0,26,3,4,61,-58,"{'id': 2301364, 'Guid': 'BVBL1037J16 1', 'Nam..."
4,Vic Huysman,7,S,False,9,10,0,0,0,0,6,1,19,-13,"{'id': 2301548, 'Guid': 'BVBL1037J18 1', 'Nam..."
5,Vic Huysman,7,S,False,4,9,0,2,0,2,-2,1,18,-20,"{'id': 458298, 'Guid': 'BVBL1037J18 1', 'Name..."
6,Vic Huysman,8,S,True,22,22,5,16,3,24,-34,3,49,-83,"{'id': 458382, 'Guid': 'BVBL1037J16 1', 'Name..."


In [93]:
game_team_df = pd.json_normalize(data['GameTeam'])
game_team_df

,id,Guid,Name,FreeThrowsGiven,FreeThrowsMade,FieldGoalsMade,ThreePointersMade,Score,LongestRun,BiggestLead,...,Game.HomeTeam.Squads.BVBL712602_BVBL714465_BVBL737616_BVBL744354_BVBL748910.Score,Game.HomeTeam.Squads.BVBL712602_BVBL714465_BVBL737616_BVBL744354_BVBL748910.Minutes,Game.HomeTeam.Squads.BVBL712602_BVBL714465_BVBL737616_BVBL744354_BVBL748910.Players.BVBL712602,Game.HomeTeam.Squads.BVBL712602_BVBL714465_BVBL737616_BVBL744354_BVBL748910.Players.BVBL714465,Game.HomeTeam.Squads.BVBL712602_BVBL714465_BVBL737616_BVBL744354_BVBL748910.Players.BVBL737616,Game.HomeTeam.Squads.BVBL712602_BVBL714465_BVBL737616_BVBL744354_BVBL748910.Players.BVBL744354,Game.HomeTeam.Squads.BVBL712602_BVBL714465_BVBL737616_BVBL744354_BVBL748910.Players.BVBL748910,Game.HomeTeam.Squads.BVBL712602_BVBL714465_BVBL737616_BVBL744354_BVBL748910.PlusMinus,Game.HomeTeam.Squads.BVBL712602_BVBL714465_BVBL737616_BVBL744354_BVBL748910.ScorePerMinute,Game.HomeTeam.Squads.BVBL712602_BVBL714465_BVBL737616_BVBL744354_BVBL748910.PlusMinusPerMinute
0,2329102,BVBL1037J18 1,BBC Haantjes Certifisc Oudenaarde J18 A,34,17,72,15,104,18,54,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,458298,BVBL1037J18 1,BBC Haantjes Certifisc Oudenaarde J18 A,4,7,54,0,61,10,15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1273562,BVBL1037J18 1,BBC Haantjes Certifisc Oudenaarde J18 A,36,15,56,3,74,10,23,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1228435,BVBL1037J16 1,BBC Haantjes Certifisc Oudenaarde J16 A,24,12,32,0,44,5,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2301364,BVBL1037J16 1,BBC Haantjes Certifisc Oudenaarde J16 A,20,14,46,6,66,7,10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,458382,BVBL1037J16 1,BBC Haantjes Certifisc Oudenaarde J16 A,26,9,32,12,53,5,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2301548,BVBL1037J18 1,BBC Haantjes Certifisc Oudenaarde J18 A,21,14,66,3,83,7,20,...,6.0,2.0,Brecht Van Glabeke,Mats Renard,Matti Bogaert,Vic Huysman,Milan De Brabander,2.0,3.0,1.0


In [192]:
# Fetch Player Data from Webpage
url = "https://app.basketballstatsvlaanderen.be/players/BVBL744354"
# url = "https://app.basketballstatsvlaanderen.be/players/BVBL711026"
url = "https://app.basketballstatsvlaanderen.be/players/BVBL711026"

# data = fetch_player_summary(url)
data2 = fetch_player(url)
# data2[data2['Number'] == 8].sort_values(by='TotalScore', ascending=False)
data2

,Name,Number,FunctionLetter,Starter,TotalMinutes,NormalizedMinutes,FreeThrows,FieldGoals,ThreePointers,TotalScore,PlusMinus,Faults,Plus,Minus,GameTeam
0,Olivier Goossens,10,S,True,13,14,0,4,0,4,34,0,52,-18,"{'id': 2328917, 'Guid': 'BVBL1034J16 1', 'Nam..."
1,Olivier Goossens,10,S,False,17,17,2,2,0,4,-5,4,26,-31,"{'id': 1183898, 'Guid': 'BVBL1034J16 1', 'Nam..."
2,Olivier Goossens,10,S,False,20,20,0,4,0,4,41,1,74,-33,"{'id': 2301285, 'Guid': 'BVBL1034J16 1', 'Nam..."


In [79]:
import pandas as pd

# test = fetch_ALL(url)
df = test.copy()
df_game_team = df['GameTeam']
# df = pd.concat([df.drop(columns=['GameTeam']), df_game_team], axis=1)
# Unnest the 'GameTeam' column so its dictionary keys become columns in df
game_team_df = pd.json_normalize(df['GameTeam'])
game_team_df

,id,Guid,Name,FreeThrowsGiven,FreeThrowsMade,FieldGoalsMade,ThreePointersMade,Score,LongestRun,BiggestLead,...,Game.HomeTeam.Squads.BVBL712602_BVBL737616_BVBL748604_BVBL748910_BVBL761196.Score,Game.HomeTeam.Squads.BVBL712602_BVBL737616_BVBL748604_BVBL748910_BVBL761196.Minutes,Game.HomeTeam.Squads.BVBL712602_BVBL737616_BVBL748604_BVBL748910_BVBL761196.Players.BVBL712602,Game.HomeTeam.Squads.BVBL712602_BVBL737616_BVBL748604_BVBL748910_BVBL761196.Players.BVBL737616,Game.HomeTeam.Squads.BVBL712602_BVBL737616_BVBL748604_BVBL748910_BVBL761196.Players.BVBL748604,Game.HomeTeam.Squads.BVBL712602_BVBL737616_BVBL748604_BVBL748910_BVBL761196.Players.BVBL748910,Game.HomeTeam.Squads.BVBL712602_BVBL737616_BVBL748604_BVBL748910_BVBL761196.Players.BVBL761196,Game.HomeTeam.Squads.BVBL712602_BVBL737616_BVBL748604_BVBL748910_BVBL761196.PlusMinus,Game.HomeTeam.Squads.BVBL712602_BVBL737616_BVBL748604_BVBL748910_BVBL761196.ScorePerMinute,Game.HomeTeam.Squads.BVBL712602_BVBL737616_BVBL748604_BVBL748910_BVBL761196.PlusMinusPerMinute
0,2329102,BVBL1037J18 1,BBC Haantjes Certifisc Oudenaarde J18 A,34,17,72,15,104,18,54,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1228435,BVBL1037J16 1,BBC Haantjes Certifisc Oudenaarde J16 A,24,12,32,0,44,5,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,458382,BVBL1037J16 1,BBC Haantjes Certifisc Oudenaarde J16 A,26,9,32,12,53,5,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1273562,BVBL1037J18 1,BBC Haantjes Certifisc Oudenaarde J18 A,36,15,56,3,74,10,23,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2301364,BVBL1037J16 1,BBC Haantjes Certifisc Oudenaarde J16 A,20,14,46,6,66,7,10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2301548,BVBL1037J18 1,BBC Haantjes Certifisc Oudenaarde J18 A,21,14,66,3,83,7,20,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,458298,BVBL1037J18 1,BBC Haantjes Certifisc Oudenaarde J18 A,4,7,54,0,61,10,15,...,2.0,1.0,Brecht Van Glabeke,Matti Bogaert,Rémi Bauters,Milan De Brabander,Sofian D'haenen,0.0,2.0,0.0


In [46]:
pd.json_normalize(test)

""
0
1
2
3
4
5
6
7
8
9


In [5]:
# Read the PLOEG.HTML file
with open(r'C:\Users\StijnHuysman\OneDrive - mateco cloud\GITHUB REPOS\HAANTJES\PLOEG.HTML', 'r', encoding='utf-8') as file:
    ploeg_html = file.read()

# Parse the HTML content
soup = BeautifulSoup(ploeg_html, 'html.parser')

# Extract player names and their corresponding href links
players = []
for a_tag in soup.find_all('a', href=True):
    if 'player' in a_tag['href']:  # Filter only player links
        players.append({'NAME': a_tag.text.strip(), 'HREF': a_tag['href']})
        # Extract SPELERID from the HREF field
        for player in players:
            player['SPELERID'] = player['HREF'].split('/')[-1]
# Create a DataFrame to display the players
# players_df = pd.DataFrame(players, columns=['NAME', 'HREF'])
# Convert the players list to a DataFrame
players_df = pd.DataFrame(players, columns=['NAME', 'HREF', 'SPELERID'])
# Remove duplicate rows based on 'NAME' and 'SPELERID' columns
players_df = players_df.drop_duplicates(subset=['NAME', 'SPELERID'])
# Display the DataFrame
players_df['HREF']
players_df['SPELERID']
# Create a new DataFrame with player summaries using fetch_player_summary for each SPELERID
player_summaries = []
for _, row in players_df.iterrows():
    player_url = f"https://app.basketballstatsvlaanderen.be/players/{row['SPELERID']}?season=2425"
    summary_df = fetch_player_summary(player_url)
    summary_df['SPELERID'] = row['SPELERID']
    summary_df['NAME'] = row['NAME']
    player_summaries.append(summary_df)

all_players_summary_df = pd.concat(player_summaries, ignore_index=True)
all_players_summary_df
for index, row in players_df.iterrows():
    print(f"Name: {row['NAME']}, SPELERID: {row['SPELERID']}")




NameError: name 'BeautifulSoup' is not defined

In [184]:
all_players_summary_df

,Name,Number,Starter %,TotalMinutes Avg,NormalizedMinutes Avg,FreeThrows Avg,FieldGoals Avg,ThreePointers Avg,TotalScore Avg,PlusMinus Avg,Faults Avg,Plus Avg,Minus Avg,SPELERID,NAME
0,Gust Ottevaere,11,0.000000,11.846154,13.461538,0.307692,3.076923,0.230769,3.615385,8.692308,1.307692,25.846154,-17.153846,BVBL750587,Gust Ottevaere
1,Jasper Espeel,11,83.333333,21.777778,25.444444,1.777778,5.777778,0.833333,8.388889,0.333333,1.666667,38.055556,-37.722222,BVBL749137,Jasper Espeel
2,Jonah Cattoir,5,69.444444,22.361111,27.388889,2.361111,11.444444,0.000000,13.805556,3.194444,2.055556,43.250000,-40.055556,BVBL762639,Jonah Cattoir
3,Marcel Heerman,12,17.857143,15.222222,17.481481,0.407407,5.777778,0.000000,6.185185,17.444444,1.666667,37.296296,-19.851852,BVBL668586,Marcel Heerman
4,Mathis Cromheeke,14,85.714286,22.000000,25.714286,0.571429,3.142857,0.285714,4.000000,-2.000000,0.904762,40.571429,-42.571429,BVBL715872,Mathis Cromheeke
5,Niels De Coussemaker,6,9.523810,11.941176,13.823529,0.352941,0.705882,0.000000,1.058824,12.058824,1.000000,25.529412,-13.470588,BVBL708695,Niels De Coussemaker
6,Tibo Despriet,12,95.833333,22.166667,26.541667,1.125000,14.333333,0.250000,15.708333,28.791667,1.833333,62.583333,-33.791667,BVBL738959,Tibo Despriet
7,Torre Beeckman,15,95.833333,21.208333,25.166667,1.333333,8.916667,1.000000,11.250000,23.416667,0.666667,56.291667,-32.875000,BVBL720212,Torre Beeckman
8,Vic Huysman,8,83.870968,21.774194,25.967742,1.516129,5.935484,0.000000,7.451613,22.161290,1.548387,55.870968,-33.709677,BVBL744354,Vic Huysman
9,Viktor L'Hommelet,5,91.304348,21.347826,25.565217,1.043478,14.869565,1.043478,16.956522,25.521739,1.434783,57.913043,-32.391304,BVBL714059,Viktor L'Hommelet


In [33]:
import requests 
import re   
import json
from bs4 import BeautifulSoup
import pandas as pd
# Fetch Player Data from Webpage
url = "https://app.basketballstatsvlaanderen.be/players/BVBL744354?season=2425"

response = requests.get(url)
html_doc = response.text

soup = BeautifulSoup(html_doc, 'html.parser')
games_data = []

for script in soup.find_all('script'):
    if script.string and "var games = [" in script.string:
        # Use a regex to find the content of the array
        match = re.search(r"var games = \[.*?\];", script.string, re.DOTALL)
        if match:
            # Extract the string content
            games_str = match.group(0).replace('var games =', '').strip(' ;')
            try:
                # Use json.loads to parse the array string into a Python list
                games_data = json.loads(games_str)
                break  # Stop once we've found and extracted the data
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON: {e}")

fields_to_keep = [
    # 'id', 'GameTeamId', 'Guid', 
    'Name', 'Number', 'FunctionLetter', 'Starter',
    'TotalMinutes', 'NormalizedMinutes', 'FreeThrows', 'FieldGoals', 'ThreePointers',
    'TotalScore', 'PlusMinus', 'Faults', 'Plus', 'Minus' ,  
    # 'GameTeam_GameId', 
    'GameTeam_GameDate', 'GameTeam_GameResult', 'GameTeam_GameGuid',
    'GameTeam_HomeTeamName', 'GameTeam_AwayTeamName'
]

filtered_games_data = [
    {k: game[k] for k in fields_to_keep if k in game}
    for game in games_data
]

filtered_games_data
# Create a DataFrame from filtered_games_data
df_games = pd.DataFrame(filtered_games_data)



# Calculate the average for the specified fields
fields_to_average = [
    'TotalMinutes', 'NormalizedMinutes', 'FreeThrows', 'FieldGoals', 'ThreePointers',
    'TotalScore', 'PlusMinus', 'Faults', 'Plus', 'Minus'
]
df_games_grouped = df_games.groupby('Name')[fields_to_average].mean()
averages = {field: df_games[field].mean() for field in fields_to_average if field in df_games.columns}
# Prepare the summary DataFrame
summary_data = []

for name, group in df_games.groupby('Name'):
    entry = {
        'Name': name,
        'Number': group['Number'].iloc[0] if 'Number' in group.columns else None,
        'Starter %': group['Starter'].mean() * 100 if 'Starter' in group.columns else None
    }
    for field in fields_to_average:
        filtered_group = group[group['TotalMinutes'] > 0]
        entry[field + ' Avg'] = filtered_group[field].mean() if field in filtered_group.columns else None
    summary_data.append(entry)

summary_df = pd.DataFrame(summary_data)
summary_df


,Name,Number,Starter %,TotalMinutes Avg,NormalizedMinutes Avg,FreeThrows Avg,FieldGoals Avg,ThreePointers Avg,TotalScore Avg,PlusMinus Avg,Faults Avg,Plus Avg,Minus Avg
0,Vic Huysman,8,83.870968,21.774194,25.967742,1.516129,5.935484,0.0,7.451613,22.16129,1.548387,55.870968,-33.709677
